<a href="https://colab.research.google.com/github/bhaskarkumar1667/sih_2026/blob/main/cnn_modeltraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 843.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/1

In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import networkx as nx
import numpy as np
import pandas as pd
import random
from collections import deque
import torch.optim as optim

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Using device: cpu


In [2]:
def build_railway_graph(num_stations=50, num_edges=100):
    """
    Creates a simulated spatial graph of railway stations and tracks.
    Node features: [current_train_count, historical_delay_avg, maintenance_status]
    """
    G = nx.gnm_random_graph(num_stations, num_edges)

    # Extract Edge Index for PyTorch Geometric
    edges = list(G.edges())
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    # Generate mock time-series data (e.g., 24 hours of traffic)
    seq_length = 24
    node_features = torch.rand((seq_length, num_stations, 3))

    # Simulated target: Congestion level/Travel time for the next hour
    y = torch.rand((num_stations, 1))

    return edge_index.to(device), node_features.to(device), y.to(device)

edge_index, X_seq, y = build_railway_graph()
print(f"Graph constructed: {X_seq.shape[1]} stations, {edge_index.shape[1]} tracks.")

Graph constructed: 50 stations, 100 tracks.


In [3]:
class TGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(TGCN, self).__init__()
        # Spatial Graph Convolution
        self.gcn = GCNConv(in_channels, hidden_channels)
        # Temporal Gated Recurrent Unit
        self.gru = torch.nn.GRUCell(hidden_channels, hidden_channels)
        # Final prediction layer (predicts congestion factor)
        self.linear = torch.nn.Linear(hidden_channels, 1)

    def forward(self, x_seq, edge_index):
        # x_seq shape: [time_steps, num_nodes, features]
        time_steps = x_seq.size(0)
        num_nodes = x_seq.size(1)
        hidden_state = torch.zeros(num_nodes, self.gru.hidden_size).to(device)

        # Process each time step through GCN then GRU
        for t in range(time_steps):
            x_t = x_seq[t]
            # Spatial dependency
            gcn_out = F.relu(self.gcn(x_t, edge_index))
            # Temporal dependency
            hidden_state = self.gru(gcn_out, hidden_state)

        # Predict future state
        out = self.linear(hidden_state)
        return torch.sigmoid(out) # Output between 0 (clear) and 1 (blocked/congested)

model = TGCN(in_channels=3, hidden_channels=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.MSELoss()

In [4]:
class RoutingDQN(torch.nn.Module):
    def __init__(self, num_nodes):
        super(RoutingDQN, self).__init__()
        # State: current_node (one-hot), destination_node (one-hot), network_congestion (from TGCN)
        input_dim = num_nodes * 2 + num_nodes
        self.fc1 = torch.nn.Linear(input_dim, 128)
        self.fc2 = torch.nn.Linear(128, 64)
        self.fc3 = torch.nn.Linear(64, num_nodes) # Q-values for next station

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return self.fc3(x) # Returns raw Q-values for action selection

def get_action(state, q_network, epsilon, valid_neighbors):
    """Epsilon-greedy selection ensuring the train only moves to physically connected tracks."""
    if random.random() < epsilon:
        return random.choice(valid_neighbors)
    else:
        with torch.no_grad():
            q_values = q_network(state)
            # Mask invalid actions (disconnected stations)
            valid_q = {n: q_values[0][n].item() for n in valid_neighbors}
            return max(valid_q, key=valid_q.get)

rl_agent = RoutingDQN(num_nodes=50).to(device)

In [5]:
epochs = 100
model.train()

for epoch in range(epochs):
    optimizer.zero_grad()

    # 1. Forward pass T-GCN to predict track congestion
    predicted_congestion = model(X_seq, edge_index)
    loss = criterion(predicted_congestion, y)

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch} | T-GCN Loss: {loss.item():.4f}")

print("\n--- Network Simulation ---")
# Simulate routing a train using the T-GCN output as the environmental penalty
start_node = 0
dest_node = 45
print(f"Routing train from Station {start_node} to {dest_node} avoiding predicted bottlenecks...")



Epoch 0 | T-GCN Loss: 0.0861
Epoch 20 | T-GCN Loss: 0.0629
Epoch 40 | T-GCN Loss: 0.0438
Epoch 60 | T-GCN Loss: 0.0278
Epoch 80 | T-GCN Loss: 0.0272

--- Network Simulation ---
Routing train from Station 0 to 45 avoiding predicted bottlenecks...


In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Convert PyTorch tensors to NumPy arrays
# Ensure gradient tracking is turned off using detach()
y_true = y.detach().cpu().numpy()
y_pred = predicted_congestion.detach().cpu().numpy()

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("--- T-GCN Evaluation Metrics ---")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"RMSE (Root Mean Square Error): {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

# Note: Since the graph was populated with random mock data in Cell 2,
# these scores will look poor until trained on real dataset tensors.

--- T-GCN Evaluation Metrics ---
MAE (Mean Absolute Error): 0.0731
RMSE (Root Mean Square Error): 0.1008
R² Score: 0.8810


In [9]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from torch_geometric.utils import to_networkx
from torch_geometric.data import Data
from IPython.display import HTML

# Reconstruct the NetworkX graph from the PyG edge_index
data = Data(edge_index=edge_index.cpu())
G = to_networkx(data, to_undirected=True)

# Generate a fixed layout so nodes don't jump around during animation
pos = nx.spring_layout(G, seed=42)

# Generate a sample shortest path (Simulating RL Agent Output)
start_node = 0
dest_node = 45
try:
    optimal_path = nx.shortest_path(G, source=start_node, target=dest_node)
except nx.NetworkXNoPath:
    # Fallback if the random graph didn't connect start to dest
    optimal_path = [0, 1, 2, 3, 4]

fig, ax = plt.subplots(figsize=(10, 8))

def update(num):
    ax.clear()

    # 1. Draw the base track network
    nx.draw(G, pos, ax=ax, node_color='lightgray', edge_color='gray', node_size=200, with_labels=False)

    # 2. Highlight the routing path up to the current step
    path_edges = list(zip(optimal_path[:num+1], optimal_path[1:num+1]))
    if path_edges:
        nx.draw_networkx_edges(G, pos, edgelist=path_edges, edge_color='blue', width=3.0, ax=ax)

    # 3. Highlight the current train position (Red Node)
    current_node = optimal_path[num]
    nx.draw_networkx_nodes(G, pos, nodelist=[current_node], node_color='red', node_size=400, ax=ax)

    ax.set_title(f"Dynamic Train Routing Simulation\nStep {num + 1}/{len(optimal_path)} | Current Station: Node {current_node}")

# Create the animation object
ani = animation.FuncAnimation(fig, update, frames=len(optimal_path), interval=800, repeat=False)

# Close the static plot to prevent duplicate rendering in Jupyter
plt.close()

# Render interactive HTML5 video in the notebook output cell
HTML(ani.to_jshtml())

/usr/local/lib/python3.13/dist-packages/torch_geometric/data/data.py:228: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  offset = offset + store.num_nodes
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/convert.py:160: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  assert node_store.num_nodes is not None
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/convert.py:161: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  for i in range(node_store.num_nodes):
